In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from abyss.glob import glob
import pandas as pd
import numpy as np

In [ ]:
# Define paths
PATH_input = glob.PATH_GENO+ "/02_per_chrom/"
PATH_output = glob.PATH_usefull + "/LD_scores/"

In [ ]:
name = "total_pop"
chrom = "22"

path_output = PATH_output+ f"/chrom_{chrom}"
os.makedirs(path_output, exist_ok=True)

In [ ]:
print(f"Finding allele frequencies of chromosome {chrom}")
output_file = "output.txt"
os.system(f"../../plink2 --bfile {PATH_input}{name}_chr_{chrom} --make-bed --out ./temp_{chrom}"+ " > " + output_file + " 2>&1")
os.system(f"../../plink2 --bfile ./temp_{chrom} --freq --out ./AF_{chrom}"+ " > " + output_file + " 2>&1")
AFs_meta = pd.read_csv(f"./AF_{chrom}.afreq", sep="\t") # convert to pandas dataframe
os.system(f"rm AF_{chrom}*")
os.system(f"rm temp_{chrom}*")
os.system(f"rm output.txt")

In [ ]:
len(AFs_meta)

In [ ]:
window_size = 1000
step_size = 5

# List to store the windows
windows = []

# Loop through the dataframe in steps of 'step_size'
for start in range(0, len(AFs_meta) - window_size + 1, step_size):
    end = start + window_size
    window = AFs_meta.iloc[start:end]
    windows.append(window)

In [ ]:
for i in range(len(windows)):
    percent = np.round((i+1)/(len(windows)+1)*100,1)
    print(f"done {percent}%")
    window = windows[i]

In [ ]:
window

In [ ]:
snp_ids_to_keep = " ,".join(list(window['ID']))

In [ ]:
snp_ids_to_keep

In [ ]:
os.system(f"../../plink2 --bfile {PATH_input}{name}_chr_{chrom} --snps {snp_ids_to_keep} --mac 50 --geno 0.05  --recode A --out ./temp_{chrom}"+ " > " + output_file + " 2>&1")

In [ ]:
to_drop = ['FID','IID','PAT','MAT','SEX', 'PHENOTYPE']
genos = pd.read_csv(f"./temp_{chrom}.raw", sep="\t")
genos = genos.fillna(2)
genos = genos.drop(to_drop, axis=1).astype(int) - 1 # Encoding it [-1, 0, 1]
genos = genos.applymap(lambda x :x if x in [-1, 0, 1] else 1)
genos = genos.T.drop_duplicates(keep='first').T

In [ ]:
genos

In [ ]:
r_squared_matrix = genos.corr() ** 2      
# Sum r^2 values for each SNP to get LD scores
ld_scores = r_squared_matrix.sum()

In [ ]:
ld_score_df = pd.DataFrame(list(ld_scores.items()), columns=['ID', 'LD_score'])
ld_score_df['ID'] = ld_score_df['ID'].str[:-2]

In [ ]:
window

In [ ]:
ld_score_df

In [ ]:
merged_data = pd.merge(ld_score_df, window, on='ID')

In [ ]:
merged_data = merged_data[['ID','LD_score']]

In [ ]:
merged_data.to_pickle(path_output + f"/window_{i}.pkl")

In [ ]:
merged_data

In [ ]:
is_in_outputs = [int(f.split("_")[1].split('.')[0]) for f in os.listdir(path_output)]
i_to_iterate = [f for f in list(range(len(windows))) if f not in is_in_outputs]
for i in i_to_iterate:
    print(i)

In [ ]:
is_in_outputs

In [ ]:
AFs_meta

# When you have the LD bins

In [ ]:
chrom = 22

In [ ]:
path_input = glob.PATH_usefull + f"/LD_scores/chrom_{chrom}"

In [ ]:
list_of_windows = os.listdir(path_input)

In [ ]:
# Sorting the list based on the numeric value after "window_"
sorted_list_of_windows = sorted(list_of_windows, key=lambda x: int(x.split('_')[1].split('.')[0]))

In [ ]:
path_window = path_input + f"/{sorted_list_of_windows[0]}"
total = pd.read_pickle(path_window)
total['counter'] = 1
for window in sorted_list_of_windows:
    path_window = path_input + f"/{window}"
    temp = pd.read_pickle(path_window)
    temp['counter'] = 1
    total = pd.merge(total, temp, on='ID', how='outer', suffixes=('_from_total', '_from_temp'))
    total['counter'] = (total['counter_from_total'] + total['counter_from_temp'])
    total['multi_old'] = total['counter_from_total']/total['counter']
    total['multi_new'] = 1/total['counter']

    begin = total[pd.isna(total['LD_score_from_temp'])].copy()
    middle = total[~pd.isna(total['LD_score_from_total']) & ~pd.isna(total['LD_score_from_temp'])].copy()
    end = total[pd.isna(total['LD_score_from_total'])].copy()
    
    begin['LD_score'] = begin['LD_score_from_total']
    begin['counter'] = begin['counter_from_total']
    begin = begin[['ID','LD_score','counter']]
    end['LD_score'] = end['LD_score_from_temp']
    end['counter'] = end['counter_from_temp']
    end = end[['ID','LD_score','counter']]
    
    middle['LD_score'] = middle['multi_old']*middle['LD_score_from_total'] + middle['multi_new']*middle['LD_score_from_temp']
    
    middle = middle[['ID','LD_score','counter']]
    total = pd.concat([begin, middle, end], ignore_index=True)

In [ ]:
unique_values = total['counter'].unique()
unique_values

In [ ]:
# Splitting the 'ID' column by ':' and expanding into separate columns
split_columns = total['ID'].str.split(':', expand=True)

# The middle number is in the second column (index 1) after splitting
total['position'] = split_columns[1]

# Optionally, convert 'position' to an integer if it's always a number
total['position'] = total['position'].astype(int)

In [ ]:
total

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your DataFrame is named total and it has columns 'position' and 'counter'
sns.scatterplot(data=total, x='position', y='counter')

# Optional: Add titles and labels
plt.title("Scatter Plot of Position vs Counter")
plt.xlabel("Position")
plt.ylabel("Counter")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your DataFrame is named total and it has columns 'position' and 'counter'
sns.scatterplot(data=total, x='position', y='LD_score')

# Optional: Add titles and labels
plt.title("Scatter Plot of Position vs LD score")
plt.xlabel("Position")
plt.ylabel("LD score")

plt.show()

In [ ]:
total

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Resetting the index to use it in the plot
total_reset = total.reset_index()

# Creating the scatter plot
sns.scatterplot(data=total_reset, x='index', y='counter')

# Optional: Add titles and labels
plt.title("Scatter Plot of Index vs Counter")
plt.xlabel("Index")
plt.ylabel("Counter")

plt.show()


In [ ]:
AFs_meta

In [ ]:
# Splitting the 'ID' column by ':' and expanding into separate columns
split_columns = AFs_meta['ID'].str.split(':', expand=True)

# The middle number is in the second column (index 1) after splitting
AFs_meta['position'] = split_columns[1]

# Optionally, convert 'position' to an integer if it's always a number
AFs_meta['position'] = AFs_meta['position'].astype(int)

In [ ]:
AFs_meta

In [ ]:
df_sorted = AFs_meta.sort_values(by='position')

In [ ]:
is_same_order = AFs_meta.index.equals(df_sorted.index)

In [ ]:
is_same_order

In [ ]:
total

In [ ]:
os.system(f"rm {path_input}/*")

In [ ]:
max_LD = np.round(total['LD_score'].max(),3)
max_LD

In [ ]:
total.to_pickle(f"{path_input}/LD_scores_max_{max_LD}.pkl")